#**Multi-Lead-ECG-for-Early-Cardiac-Disease-Detection**


##**Attention-Based CNN for ECG Signal Classification**

AI-Based Reconstruction and Classification of Multi-Lead ECG Signals from Reduced Lead Data for Early Cardiac Disease Detection

The objective is to develop a robust neural network model capable of identifying patterns in ECG signals and accurately classifying them into different diagnostic categories.


# **Installations & Imports & Dataset Loading**

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import ast

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

path = '/kaggle/input/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/'
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

# Filter rows where 'diagnostic_superclass' has either no label or multiple labels
mask = Y['diagnostic_superclass'].apply(lambda x: len(x) == 1)
Y_filtered = Y[mask]

# Now, you should also filter the raw signal data 'X' to match the updated labels
X_filtered = X[mask]

# Continue to split data into train and test
test_fold = 10

# Train
X_train = X_filtered[np.where(Y_filtered.strat_fold != test_fold)]
y_train = Y_filtered[(Y_filtered.strat_fold != test_fold)].diagnostic_superclass

# Test
X_test = X_filtered[np.where(Y_filtered.strat_fold == test_fold)]
y_test = Y_filtered[Y_filtered.strat_fold == test_fold].diagnostic_superclass

# To simplify y_train and y_test further, convert the list entries to single string labels
y_train = y_train.apply(lambda x: x[0])
y_test = y_test.apply(lambda x: x[0])

# **Label Distribution**

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

# **ECG Visualization**

In [ ]:
import matplotlib.pyplot as plt

def plot_ecg_examples(X, Y, num_examples=5):
    # Randomly select some examples
    idx = np.random.choice(np.arange(len(X)), num_examples, replace=False)

    for i in idx:
        plt.figure(figsize=(10, 3))
        plt.plot(X[i].T)  # If your ECG data has multiple channels, they'll all be plotted.
        plt.title(f"Diagnostic Superclass: {', '.join(Y.iloc[i])}")
        plt.xlabel('Time')
        plt.ylabel('Amplitude')
        plt.grid(True)
        plt.show()

# To visualize 5 random examples from the training set:
plot_ecg_examples(X_train, y_train, num_examples=5)

# **Single ECG Visualization**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_single_ecg_example(X, Y):
    # Randomly select one example
    idx = np.random.choice(np.arange(len(X)))

    sample_ecg = X[idx]
    label = Y.iloc[idx]

    # Create a 12x1 grid of plots
    fig, axs = plt.subplots(nrows=12, ncols=1, figsize=(10, 15))

    for i in range(12):
        sns.lineplot(x=np.arange(sample_ecg.shape[1]), y=sample_ecg[i], ax=axs[i])
        axs[i].set_title(f"Channel {i+1}")
        axs[i].set_ylabel('Amplitude')
        axs[i].grid(True)

    plt.tight_layout()
    plt.suptitle(f"Diagnostic Superclass: {', '.join(label)}", y=1.02)
    plt.show()

# To visualize a random example from the training set:
plot_single_ecg_example(X_train, y_train)

# **Data Preprocessing**

In [ ]:
X_train = np.array(X_train)
X_test = np.array(X_test)

# If y_train and y_test are lists of labels, convert them to categorical format
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

# Initialize the LabelEncoder
le = LabelEncoder()

# Combine train and test labels
all_labels = pd.concat([y_train, y_test])

# Fit on combined labels
le.fit(all_labels)

# Transform separately
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

y_train_cat = to_categorical(y_train_enc)
y_test_cat = to_categorical(y_test_enc)

In [ ]:
# Unique labels and their counts in the training set
train_label_counts = y_train.value_counts()

# Unique labels and their counts in the test set
test_label_counts = y_test.value_counts()

print("Training Labels:")
print(train_label_counts)

print("\nTest Labels:")
print(test_label_counts)

In [ ]:
input_shape = (X_train.shape[1], X_train.shape[2])  # (number_of_time_points, 12)
input_shape

# **Model Development & Hyperparameter Tuning**

In [ ]:
!pip install keras-tuner

wandbimport tensorflow as tf
import wandb
from wandb.keras import WandbCallback
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation, LeakyReLU
from kerastuner.tuners import RandomSearch
from tensorflow.keras.callbacks import EarlyStopping  # Added import
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import Callback  # Corrected import


# 1. Set up Weights & Biases for logging
wandb.init(project="EKG-CONV1D")

# Define the EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# Define a list of different learning rates to search through
learning_rates = [1e-2, 1e-3, 1e-4]

import gc  # for garbage collection

def build_model(hp):
    model = Sequential()

    # Input Layer
    activation_1 = hp.Choice('activation_1', values=['relu', 'leaky_relu'])
    model.add(Conv1D(filters=hp.Int('filters_1', 16, 256, step=32),
                     kernel_size=hp.Int('kernel_size_1', 2, 8, step=1),
                     input_shape=input_shape))
    if activation_1 == "leaky_relu":
        model.add(LeakyReLU())
    else:
        model.add(Activation(activation_1))
    model.add(MaxPooling1D(pool_size=2))

    # Additional Convolutional Layers
    for i in range(hp.Int('num_conv_layers', 1, 3)):
        activation_conv = hp.Choice(f'activation_conv_{i+2}', values=['relu', 'tanh', 'sigmoid', 'leaky_relu'])
        model.add(Conv1D(filters=hp.Int(f'filters_{i+2}', 16, 256, step=32),
                         kernel_size=hp.Int(f'kernel_size_{i+2}', 2, 8, step=2)))
        if activation_conv == "leaky_relu":
            model.add(LeakyReLU())
        else:
            model.add(Activation(activation_conv))
        model.add(MaxPooling1D(pool_size=2))

    model.add(Flatten())

    # Dense Layers
    activation_dense_1 = hp.Choice('activation_dense_1', values=['relu', 'tanh', 'sigmoid', 'leaky_relu'])
    model.add(Dense(hp.Int('dense_units_1', 32, 512, step=32)))
    if activation_dense_1 == "leaky_relu":
        model.add(LeakyReLU())
    else:
        model.add(Activation(activation_dense_1))
    model.add(Dropout(hp.Float('dropout_1', 0.0, 0.5, step=0.1)))

    for i in range(hp.Int('num_dense_layers', 1, 2)):
        activation_dense = hp.Choice(f'activation_dense_{i+2}', values=['relu', 'tanh', 'sigmoid', 'leaky_relu'])
        model.add(Dense(hp.Int(f'dense_units_{i+2}', 32, 512, step=32)))
        if activation_dense == "leaky_relu":
            model.add(LeakyReLU())
        else:
            model.add(Activation(activation_dense))
        model.add(Dropout(hp.Float(f'dropout_{i+2}', 0.0, 0.5, step=0.1)))

    # Output Layer
    model.add(Dense(y_train_cat.shape[1], activation='softmax'))

    # Compiling with a choice of learning rate
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Choice('learning_rate', learning_rates)),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

# Custom callback for memory management
class MemoryCleanerCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()
    def on_train_end(self, logs=None):
        gc.collect()


# 3. Set up RandomSearch with KerasTuner
tuner = RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=64,
    executions_per_trial=1,
    directory='random_search_logs',
    project_name='kerastuner_wandb_integration')

# Add the MemoryCleanerCallback to the list of callbacks
tuner.search(X_train, y_train_cat,
             epochs=100,
             batch_size=32,
             validation_split=0.2,
             callbacks=[WandbCallback(save_model=False), early_stopping])

# **Model Training**

In [ ]:
# 5. Retrain best model to get the history
best_model = tuner.get_best_models(num_models=1)[0]
history = best_model.fit(X_train, y_train_cat, epochs=10, validation_split=0.2).history

plt.figure(figsize=(10, 6))
plt.plot(history['accuracy'], label='Accuracy')
plt.plot(history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Save plot to a file
plot_filename = "accuracy_plot.png"
plt.savefig(plot_filename)
plt.show()

# 6. Log test metrics and plot to wandb
test_loss, test_accuracy = best_model.evaluate(X_test, y_test_cat)
wandb.log({'test_loss': test_loss, 'test_accuracy': test_accuracy, 'Accuracy Plot': wandb.Image(plot_filename)})

# 7. Save the best model and display its parameters
best_model.save("best_model.h5")
print("Best Model Parameters:", tuner.oracle.get_best_trials(num_trials=1)[0].hyperparameters.values)
wandb.finish()

# **Experiment Tracking**

Weights & Biases (WandB) is used to track model training experiments.

This tool records metrics such as:

- Training loss
- Validation loss
- Accuracy
- Hyperparameters

The experiment logs are compressed and saved for future reference.

In [ ]:
!zip -r wandb.zip ./wandb/

In [ ]:
from IPython.display import FileLink
FileLink(r'wandb.zip')

In [ ]:
from tensorflow.keras.layers import MultiHeadAttention

def build_model(hp):
    model = Sequential()

    # Input Layer
    activation_1 = hp.Choice('activation_1', values=['relu', 'tanh', 'sigmoid', 'leaky_relu'])
    model.add(Conv1D(filters=hp.Int('filters_1', 16, 256, step=32),
                     kernel_size=hp.Int('kernel_size_1', 2, 8, step=1),
                     input_shape=input_shape))
    if activation_1 == "leaky_relu":
        model.add(LeakyReLU())
    else:
        model.add(Activation(activation_1))
    model.add(MaxPooling1D(pool_size=2))

    # Additional Convolutional Layers
    for i in range(hp.Int('num_conv_layers', 1, 3)):
        activation_conv = hp.Choice(f'activation_conv_{i+2}', values=['relu', 'tanh', 'sigmoid', 'leaky_relu'])
        model.add(Conv1D(filters=hp.Int(f'filters_{i+2}', 16, 256, step=32),
                         kernel_size=hp.Int(f'kernel_size_{i+2}', 2, 8, step=2)))
        if activation_conv == "leaky_relu":
            model.add(LeakyReLU())
        else:
            model.add(Activation(activation_conv))
        model.add(MaxPooling1D(pool_size=2))

    # Add Multi-Head Attention Mechanism
    model.add(MultiHeadAttention(num_heads=hp.Int('num_heads', 2, 8, step=2), key_dim=hp.Int('key_dim', 32, 128, step=32)))
    model.add(Flatten())

    # Dense Layers
    activation_dense_1 = hp.Choice('activation_dense_1', values=['relu', 'tanh', 'sigmoid', 'leaky_relu'])
    model.add(Dense(hp.Int('dense_units_1', 32, 512, step=32)))
    if activation_dense_1 == "leaky_relu":
        model.add(LeakyReLU())
    else:
        model.add(Activation(activation_dense_1))
    model.add(Dropout(hp.Float('dropout_1', 0.0, 0.5, step=0.1)))

    for i in range(hp.Int('num_dense_layers', 1, 2)):
        activation_dense = hp.Choice(f'activation_dense_{i+2}', values=['relu', 'tanh', 'sigmoid', 'leaky_relu'])
        model.add(Dense(hp.Int(f'dense_units_{i+2}', 32, 512, step=32)))
        if activation_dense == "leaky_relu":
            model.add(LeakyReLU())
        else:
            model.add(Activation(activation_dense))
            model.add(Dropout(hp.Float(f'dropout_{i+2}', 0.0, 0.5, step=0.1)))

    # Output Layer
    model.add(Dense(y_train_cat.shape[1], activation='softmax'))

    # Compiling with a choice of learning rate
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Choice('learning_rate', learning_rates)),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model